1. ЗАГРУЗКА И ПОДГОТОВКА ДАННЫХ

In [ ]:
'''Ячейка номер: 1.1. Назначение: Импортируем зависимости которые понадобятся для дальнейшей работы'''

import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
#from pycaret.clustering import *
#from sklearn.datasets import make_blobs
#mpl.rcParams['figure.dpi'] = 100 #300 сделает картинки большими

from IPython.display import display, clear_output
from ipywidgets import Dropdown
import ipywidgets as widgets
from ipywidgets import IntSlider
import io


import warnings
warnings.filterwarnings("ignore")

pd.set_option('display.max_columns', None)

 # Глобальная переменная
#data_tmp = None 
#list_of_feature = None
#city_cut = None

In [ ]:
'''
Ячейка номер: 1.2. Назначение: Импортируем и преобразуем в нужные форматы данные по On-Shelf Availability (OSA). Это параметр который представляет собой соотношение кол-ва продуктов которые фактически были на полке 
 полках в торговой точке в момент визита, к количеству продуктов которое потенциально могло там быть. 
 Структура данных в загружаемых файлах: 
 VISIT_DATE - дата визита в торговую точку в формате 2026-07-01
 PHOTO_AUDIT_ID - уникальный ID визита в торговую точку вида 3006172_1146148_20260701_2
 SHIP_TO - уникальный ID торговой точки вида 850023988
 OSA - показатель   On-Shelf Availability измеренный в момент визита. Число от 0 до 1.
 '''

osa_path = r'C:\Users\rokotyev\Yandex.Disk\_Main Data Rokotyan\2. Проекты\60. Подбор пар ТТ для тестов\_data\osa'
df_osa = pd.DataFrame()
for filename in os.listdir(osa_path):
    df_tmp = pd.read_csv(osa_path + '\\' + filename, sep=';', dtype='str')
    df_osa = pd.concat([df_osa, df_tmp])
    df_tmp = pd.DataFrame()


df_osa['VISIT_DATE'] = pd.to_datetime(df_osa['VISIT_DATE'])
df_osa[['SHIP_TO']] = df_osa[['SHIP_TO']].astype('int')
df_osa[['OSA']] = df_osa[['OSA']].astype('float')
#
df_osa['YEAR'] = df_osa['VISIT_DATE'].dt.year
df_osa['MONTH'] = df_osa['VISIT_DATE'].dt.month
df_osa['WEEK'] = df_osa['VISIT_DATE'].dt.isocalendar().week
df_osa[['YEAR', 'MONTH','WEEK']] = df_osa[['YEAR', 'MONTH','WEEK']].astype('int')
df_osa['YEAR_MONTH'] = ''
df_osa['YEAR_WEEK'] = ''
df_osa.loc[df_osa['MONTH'] < 10, 'YEAR_MONTH'] = '0'
df_osa.loc[df_osa['WEEK'] < 10, 'YEAR_WEEK'] = '0'
df_osa['YEAR_MONTH'] = df_osa['YEAR'].astype('str') + df_osa['YEAR_MONTH'] + df_osa['MONTH'].astype('str') 
df_osa['YEAR_MONTH'] = df_osa['YEAR_MONTH'].astype('int')

df_osa['YEAR_WEEK'] = df_osa['YEAR'].astype('str') + df_osa['YEAR_WEEK'] + df_osa['WEEK'].astype('str') 
df_osa['YEAR_WEEK'] = df_osa['YEAR_WEEK'].astype('int')

#df_osa.head(2)

In [ ]:
'''
Ячейка номер: 1.3. Назначение: Импортируем и преобразуем в нужные форматы данные по объему продаж торговой точки за неделю

Обновлено: данные теперь читаются из двух новых источников - sales_sellin (отгрузки поставщика в торговую точку) и 
sales_sellout (продажи из торговой точки конечному покупателю). У обоих источников одинаковый набор ключевых полей 
(SHIP_TO, YEAR_WEEK), но объём продаж в sales_sellout представлен в рублях (колонка CAF), а не в тысячах рублей 
(kCAF), как везде в проекте, поэтому для sales_sellout добавлен пересчёт CAF в kCAF (деление на 1000).

ВНИМАНИЕ, нюанс на который стоит обратить внимание: если для одной и той же точки и недели есть данные и в 
sales_sellin, и в sales_sellout, то в df_sales окажутся ДВЕ строки с одинаковыми SHIP_TO и YEAR_WEEK. Дальше по 
коду (ячейка 3.1) итоговое значение kCAF за период считается как среднее по неделям - соответственно для такой 
недели sellin и sellout усреднятся между собой. Если это не то поведение, которое нужно (например, если для 
конкретной точки должен использоваться только один источник, либо значения должны суммироваться, а не 
усредняться) - напишите, поправим логику агрегации.

 Структура данных в загружаемых файлах: 
"SHIP_TO" - уникальный ID торговой точки вида 850023988
"YEAR_WEEK" - номер недели в формате 202602, где первые 4 цифры это год, а последние 2 цифры это номер недели в году
"kCAF" - объем продаж за неделю в тысячах рублей (в sales_sellin уже в тысячах, в sales_sellout пересчитывается из рублей)
 '''

df_sales = pd.DataFrame()

sales_path_sellin = r'C:\Users\rokotyev\Yandex.Disk\_Main Data Rokotyan\2. Проекты\60. Подбор пар ТТ для тестов\_data\sales_sellin'
for filename in os.listdir(sales_path_sellin):
    df_tmp = pd.read_csv(sales_path_sellin + '\\' + filename, sep=',', dtype='str')
    df_tmp[['SHIP_TO', 'YEAR_WEEK']] = df_tmp[['SHIP_TO', 'YEAR_WEEK']].astype('int')
    df_tmp['kCAF'] = df_tmp['kCAF'].astype('float')
    df_sales = pd.concat([df_sales, df_tmp])
    df_tmp = pd.DataFrame()

sales_path_sellout = r'C:\Users\rokotyev\Yandex.Disk\_Main Data Rokotyan\2. Проекты\60. Подбор пар ТТ для тестов\_data\sales_sellout'
for filename in os.listdir(sales_path_sellout):
    df_tmp = pd.read_csv(sales_path_sellout + '\\' + filename, sep=',', dtype='str')
    df_tmp[['SHIP_TO', 'YEAR_WEEK']] = df_tmp[['SHIP_TO', 'YEAR_WEEK']].astype('int')
    df_tmp['kCAF'] = df_tmp['CAF'].astype('float')/1000
    df_tmp.drop(columns=['CAF'], inplace=True)
    df_sales = pd.concat([df_sales, df_tmp])
    df_tmp = pd.DataFrame()

#df_sales.head(10)

In [ ]:
'''
Ячейка номер: 1.4. Назначение: Импортируем и преобразуем в нужные форматы данные по проведенным визитам в торговую точку 
 Структура данных в загружаемых файлах: 
 VISIT_DATE - дата визита в торговую точку в формате 2026-07-01
 SHIP_TO - уникальный ID торговой точки вида 850023988
 ROUTE_ID - уникальный ID маршрута в рамках которого была посещена данная торговая точка, например 2496
 AGENCY_NAME - название агенства, сотрудники которого осуществляли визит например LeaderTeam
 PHOTO_AUDIT_ID - уникальный ID визита в торговую точку вида 3006172_1146148_20260701_2
 '''

visits_path = r'C:\Users\rokotyev\Yandex.Disk\_Main Data Rokotyan\2. Проекты\60. Подбор пар ТТ для тестов\_data\visits'
df_visits = pd.DataFrame()
for filename in os.listdir(visits_path):
    df_tmp = pd.read_csv(visits_path + '\\' + filename, sep=';', dtype='str')
    df_visits = pd.concat([df_visits, df_tmp])
    df_tmp = pd.DataFrame()
    
df_visits['VISIT_DATE'] = pd.to_datetime(df_visits['VISIT_DATE'])
df_visits['SHIP_TO'] = df_visits['SHIP_TO'].astype('int')

df_visits['YEAR'] = df_visits['VISIT_DATE'].dt.year
df_visits['MONTH'] = df_visits['VISIT_DATE'].dt.month
df_visits['WEEK'] = df_visits['VISIT_DATE'].dt.isocalendar().week
df_visits[['YEAR', 'MONTH', 'WEEK']] = df_visits[['YEAR', 'MONTH','WEEK']].astype('int')
df_visits['YEAR_MONTH'] = ''
df_visits['YEAR_WEEK'] = ''
df_visits.loc[df_visits['MONTH'] < 10, 'YEAR_MONTH'] = '0'
df_visits.loc[df_visits['WEEK'] < 10, 'YEAR_WEEK'] = '0'
df_visits['YEAR_MONTH'] = df_visits['YEAR'].astype('str') + df_visits['YEAR_MONTH'] + df_visits['MONTH'].astype('str') 
df_visits['YEAR_MONTH'] = df_visits['YEAR_MONTH'].astype('int')
df_visits['YEAR_WEEK'] = df_visits['YEAR'].astype('str') + df_visits['YEAR_WEEK'] + df_visits['WEEK'].astype('str') 
df_visits['YEAR_WEEK'] = df_visits['YEAR_WEEK'].astype('int')

## Блок аггрегации

df_visits = df_visits.groupby(by=['YEAR_WEEK', 'SHIP_TO']).agg({'PHOTO_AUDIT_ID':'size'}).reset_index()
df_visits.rename(columns={'PHOTO_AUDIT_ID': 'PHOTO_AUDITS_PER_WEEK'}, inplace=True)


#df_visits.head(2)

In [ ]:
'''
Ячейка номер: 1.5. Назначение: Импортируем и преобразуем в нужные форматы данные по оценке уровня мерчендайзинга в торговой точке 
 Структура данных в загружаемых файлах: 
 VISIT_DATE - дата визита в торговую точку в формате 2026-07-01
 PHOTO_AUDIT_ID - уникальный ID визита в торговую точку вида 3006172_1146148_20260701_2
 SHIP_TO - уникальный ID торговой точки вида 850023988
 PICOS_SCORE_FACT - фактическая оценка уровня мерчендайзинга в торговой точке в момент визита, число от 0 до 100
 PICOS_TARGET_SCORE - плановая оценка уровня мерчендайзинга в торговой точке, число от 0 до 100
 Например, строка может выглядеть так 2026-02-01;6000103_1548835_20260201_2;850197619;94.457918552;94.75
 '''

picos_path = r'C:\Users\rokotyev\Yandex.Disk\_Main Data Rokotyan\2. Проекты\60. Подбор пар ТТ для тестов\_data\picos'
df_picos = pd.DataFrame()
for filename in os.listdir(picos_path):
    df_tmp = pd.read_csv(picos_path + '\\' + filename, sep=';', dtype='str')
    df_picos = pd.concat([df_picos, df_tmp])
    df_tmp = pd.DataFrame()
    
df_picos['VISIT_DATE'] = pd.to_datetime(df_picos['VISIT_DATE'])
df_picos['SHIP_TO'] = df_picos['SHIP_TO'].astype('int')
df_picos['PICOS_SCORE_FACT'] = df_picos['PICOS_SCORE_FACT'].astype('float').astype('int')
df_picos['YEAR'] = df_picos['VISIT_DATE'].dt.year
df_picos['MONTH'] = df_picos['VISIT_DATE'].dt.month
df_picos['WEEK'] = df_picos['VISIT_DATE'].dt.isocalendar().week
df_picos[['YEAR', 'MONTH', 'WEEK']] = df_picos[['YEAR', 'MONTH','WEEK']].astype('int')
df_picos['YEAR_MONTH'] = ''
df_picos['YEAR_WEEK'] = ''
df_picos.loc[df_picos['MONTH'] < 10, 'YEAR_MONTH'] = '0'
df_picos.loc[df_picos['WEEK'] < 10, 'YEAR_WEEK'] = '0'
df_picos['YEAR_MONTH'] = df_picos['YEAR'].astype('str') + df_picos['YEAR_MONTH'] + df_picos['MONTH'].astype('str') 
df_picos['YEAR_MONTH'] = df_picos['YEAR_MONTH'].astype('int')
df_picos['YEAR_WEEK'] = df_picos['YEAR'].astype('str') + df_picos['YEAR_WEEK'] + df_picos['WEEK'].astype('str') 
df_picos['YEAR_WEEK'] = df_picos['YEAR_WEEK'].astype('int')

#df_picos.head(2)

In [ ]:
'''
Ячейка номер: 1.6. Назначение: Импортируем и преобразуем в нужные форматы данные по фейсингу (кол-во экземпляров товара на полке)
 Структура данных в загружаемых файлах: 

VISIT_DATE;PHOTO_AUDIT_ID;SHIP_TO;GROUP_FACT

 VISIT_DATE - дата визита в торговую точку в формате 2026-07-01
 PHOTO_AUDIT_ID - уникальный ID визита в торговую точку вида 3006172_1146148_20260701_2
 SHIP_TO - уникальный ID торговой точки вида 850023988
 GROUP_FACT - фейсинг, кол-во экземпляров товара на полке. Число больше 0.
 Например, строка может выглядеть так "2026-07-01;3004816_1000021_20260701_2;850025517;161.0"
 '''


facing_path = r'C:\Users\rokotyev\Yandex.Disk\_Main Data Rokotyan\2. Проекты\60. Подбор пар ТТ для тестов\_data\facing'
df_facing_fact = pd.DataFrame()
for filename in os.listdir(facing_path):
    df_tmp = pd.read_csv(facing_path + '\\' + filename, sep=';', dtype='str')
    df_facing_fact = pd.concat([df_facing_fact, df_tmp])
    df_tmp = pd.DataFrame()
    
df_facing_fact['VISIT_DATE'] = pd.to_datetime(df_facing_fact['VISIT_DATE'])
df_facing_fact['SHIP_TO'] = df_facing_fact['SHIP_TO'].astype('int')
df_facing_fact['GROUP_FACT'] = df_facing_fact['GROUP_FACT'].astype('float').astype('int')
df_facing_fact['YEAR'] = df_facing_fact['VISIT_DATE'].dt.year
df_facing_fact['MONTH'] = df_facing_fact['VISIT_DATE'].dt.month
df_facing_fact['WEEK'] = df_facing_fact['VISIT_DATE'].dt.isocalendar().week
df_facing_fact[['YEAR', 'MONTH', 'WEEK']] = df_facing_fact[['YEAR', 'MONTH','WEEK']].astype('int')
df_facing_fact['YEAR_MONTH'] = ''
df_facing_fact['YEAR_WEEK'] = ''
df_facing_fact.loc[df_facing_fact['MONTH'] < 10, 'YEAR_MONTH'] = '0'
df_facing_fact.loc[df_facing_fact['WEEK'] < 10, 'YEAR_WEEK'] = '0'
df_facing_fact['YEAR_MONTH'] = df_facing_fact['YEAR'].astype('str') + df_facing_fact['YEAR_MONTH'] + df_facing_fact['MONTH'].astype('str') 
df_facing_fact['YEAR_MONTH'] = df_facing_fact['YEAR_MONTH'].astype('int')
df_facing_fact['YEAR_WEEK'] = df_facing_fact['YEAR'].astype('str') + df_facing_fact['YEAR_WEEK'] + df_facing_fact['WEEK'].astype('str') 
df_facing_fact['YEAR_WEEK'] = df_facing_fact['YEAR_WEEK'].astype('int')

#df_facing_fact.head(2)

In [ ]:
'''
Ячейка номер: 1.7. Назначение: Импортируем и преобразуем в нужные форматы данные по доле полки на холодной витрине

ВНИМАНИЕ, что было исправлено в этой ячейке по сравнению с исходным файлом (это были ошибки, не новая логика):
1. В цикле os.listdir(...) и в пути к файлу вместо переменной shelfshare_path (путь к папке) по ошибке
   использовалась переменная df_shelfshare (пустой датафрейм) - из-за этого ячейка вообще не смогла бы 
   прочитать ни одного файла с диска.
2. Колонка GROUP_FACT (фейсинг) заполнялась значениями из другого датафрейма df_facing_fact - это чужой 
   показатель, к доле полки отношения не имеющий. Убрал эту строку и вместо этого привожу к нужному типу 
   "родную" колонку SHELF_SHARE, которая и есть доля полки.
3. Колонка YEAR_MONTH заполнялась значением из df_facing_fact вместо собственного рассчитанного значения.

 Структура данных в загружаемых файлах: 

VISIT_DATE;PHOTO_AUDIT_ID;SHIP_TO; SHELF_SHARE

 VISIT_DATE - дата визита в торговую точку в формате 2026-07-01
 PHOTO_AUDIT_ID - уникальный ID визита в торговую точку вида 3006172_1146148_20260701_2
 SHIP_TO - уникальный ID торговой точки вида 850023988
 SHELF_SHARE - Доля полки на холодной витрине. Число от 0 до 1.
 Например, строка может выглядеть так "2026-07-01;3004816_1000021_20260701_2;850025517; 0.8"
 '''


shelfshare_path = r'C:\Users\rokotyev\Yandex.Disk\_Main Data Rokotyan\2. Проекты\60. Подбор пар ТТ для тестов\_data\shelfshare'
df_shelfshare = pd.DataFrame()
for filename in os.listdir(shelfshare_path):
    df_tmp = pd.read_csv(shelfshare_path + '\\' + filename, sep=';', dtype='str')
    df_shelfshare = pd.concat([df_shelfshare, df_tmp])
    df_tmp = pd.DataFrame()
    
df_shelfshare['VISIT_DATE'] = pd.to_datetime(df_shelfshare['VISIT_DATE'])
df_shelfshare['SHIP_TO'] = df_shelfshare['SHIP_TO'].astype('int')
df_shelfshare['SHELF_SHARE'] = df_shelfshare['SHELF_SHARE'].astype('float')
df_shelfshare['YEAR'] = df_shelfshare['VISIT_DATE'].dt.year
df_shelfshare['MONTH'] = df_shelfshare['VISIT_DATE'].dt.month
df_shelfshare['WEEK'] = df_shelfshare['VISIT_DATE'].dt.isocalendar().week
df_shelfshare[['YEAR', 'MONTH', 'WEEK']] = df_shelfshare[['YEAR', 'MONTH','WEEK']].astype('int')
df_shelfshare['YEAR_MONTH'] = ''
df_shelfshare['YEAR_WEEK'] = ''
df_shelfshare.loc[df_shelfshare['MONTH'] < 10, 'YEAR_MONTH'] = '0'
df_shelfshare.loc[df_shelfshare['WEEK'] < 10, 'YEAR_WEEK'] = '0'
df_shelfshare['YEAR_MONTH'] = df_shelfshare['YEAR'].astype('str') + df_shelfshare['YEAR_MONTH'] + df_shelfshare['MONTH'].astype('str') 
df_shelfshare['YEAR_MONTH'] = df_shelfshare['YEAR_MONTH'].astype('int')
df_shelfshare['YEAR_WEEK'] = df_shelfshare['YEAR'].astype('str') + df_shelfshare['YEAR_WEEK'] + df_shelfshare['WEEK'].astype('str') 
df_shelfshare['YEAR_WEEK'] = df_shelfshare['YEAR_WEEK'].astype('int')

#df_shelfshare.head(2)

In [ ]:
'''
Ячейка номер: 1.8. Назначение: Импортируем и преобразуем в нужные форматы данные по доле рынка (доле в продажах) для каждой торговой точки

Обновлено: изменён путь к папке с исходными файлами (данные теперь лежат в подпапке marketshare_csv). Также 
изменилась структура файлов - поле CLIENT из данных убрано, остались только YEAR_WEEK, SHIP_TO, MARKET_SHARE.

Проверил остальной код ноутбука: колонка CLIENT из df_marketshare нигде дальше не использовалась - она нигде 
не участвовала ни в фильтрации, ни в объединении с другими датафреймами. Единственная колонка CLIENT, которая 
реально участвует в подборе точек - это CLIENT из df_pos_list (ячейка 1.9, используется в ячейке 2.2 для 
фильтрации по вывеске торговой точки) - это совершенно другая, не связанная с этой колонка. Так что удаление 
CLIENT из marketshare никак не повлияет на работу приложения.

 Структура данных в загружаемых файлах: 
 YEAR_WEEK - неделя года в формате "202543" где первые 4 цифры это год, а последние 2 цифры это номер недели в году
 SHIP_TO - уникальный ID торговой точки вида 850023988
 MARKET_SHARE - Доля рынка (доля в продажах) торговой точки. Число от 0 до 1.
 '''


marketshare_path = r'C:\Users\rokotyev\Yandex.Disk\_Main Data Rokotyan\2. Проекты\60. Подбор пар ТТ для тестов\_data\marketshare\marketshare_csv'
df_marketshare = pd.DataFrame()
for filename in os.listdir(marketshare_path):
    df_tmp = pd.read_csv(marketshare_path + '\\' + filename, sep=';', dtype='str')
    df_marketshare = pd.concat([df_marketshare, df_tmp])
    df_tmp = pd.DataFrame()

df_marketshare[['SHIP_TO', 'YEAR_WEEK']] = df_marketshare[['SHIP_TO', 'YEAR_WEEK']].astype('int')
df_marketshare['MARKET_SHARE'] = df_marketshare['MARKET_SHARE'].astype('float') * 100

#df_marketshare.head(2)

In [ ]:
'''
Ячейка номер: 1.9. Назначение: Импортируем и преобразуем в нужные форматы данные по торговым точкам
Файл содержит множество полей, но в данном проекте будут использованы следующие:  
'RSD' - Дивизион к которому относится данная точка. Высший уровень иерархии для торговых точек.
'BUSINESS_UNIT' - бизнес единицы к которой относится данная торговая точка. Уровень иерархии торговых точек, ниже чем RSD, например CENTER-SOUTH
'SALES_GROUP' - бизнес единицы к которой относится данная торговая точка. Уровень иерархии торговых точек, ниже чем BUSINESS_UNIT, например MOSCOW
'CLIENT' - вывеска торговой точки, например "VERNIY", например "PERM"
'CUSTOMER_GROUP' - формат торговой точки, например "CONVENIENCE M"
'CITY' - город в котором находится торговая точка, например "Москва"
'fid' - еще один код торговой точки в формате 1070785. Это внутренний код точки в другой системе.
'LATITUDE', 'LONGITUDE' - широта и долгота торговой точке.
 '''

df_pos_list = pd.read_csv(r"C:\Users\rokotyev\Yandex.Disk\_Main Data Rokotyan\2. Проекты\60. Подбор пар ТТ для тестов\_data\dim_poslist.csv", sep=';', dtype='str')
df_pos_list = df_pos_list.query("CHANNEL in ('NATIONAL KEY ACCOUNT', 'LOCAL KEY ACCOUNT')")
df_pos_list[['SHIP_TO', 'fid']] = df_pos_list[['SHIP_TO', 'fid']].astype('int')
df_pos_list[['LATITUDE', 'LONGITUDE']] = df_pos_list[['LATITUDE', 'LONGITUDE']].astype('float')

dict_replace_customergroup = {
'CONVENIENCE S':'CONVENIENCE S-M-L',  
'CONVENIENCE M':'CONVENIENCE S-M-L',     
'CONVENIENCE L':'CONVENIENCE S-M-L',       
'SUPERMARKET S':'SUPERMARKET S-M-L',
'SUPERMARKET M':'SUPERMARKET S-M-L',       
'SUPERMARKET L':'SUPERMARKET S-M-L',        
'HYPERMARKET S':'HYPERMARKET S-L',  
'HYPERMARKET L':'HYPERMARKET S-L'
}    
df_pos_list['CUSTOMER_GROUP'] = df_pos_list['CUSTOMER_GROUP'].replace(dict_replace_customergroup)

#print(df_pos_list.shape)
#df_pos_list.dtypes

In [173]:
'''Ячейка номер: 1.10. Назначение: Виджет в котором можно задать название проекта. В расчетах эти даты не учавствуют но выводятся в итоговый датафрейм df_pair_list '''
widget_project_name = widgets.Text(
                            placeholder='Указать название проекта',
                            description='Проект:',
                            disabled=False   
                            )

display(widget_project_name)

Text(value='', description='Проект:', placeholder='Указать название проекта')

In [174]:
'''Ячейка номер: 1.11. Назначение: Виджеты в которых можно задать даты действия проекта. В расчетах эти даты не учавствуют но выводятся в итоговый датафрейм df_pair_list  '''
widget_project_start_date = widgets.DatePicker(
                                              description='Начало:',
                                              disabled=False,
                                            )
widget_project_end_date = widgets.DatePicker(
                                              description='Окончание:',
                                              disabled=False, 
                                            )

widgets.HBox([widget_project_start_date, widget_project_end_date])

-----

2. ЗАГРУЗКА СПИСКА ТЕСТОВЫХ ТОЧЕК И ВЫБОР ПАРАМЕТРОВ ПОДБОРА ПАР

In [ ]:
'''
Ячейка номер: 2.0. Назначение: Виджет выбора режима работы приложения.

Режим "Подбор по всей АКБ" - как раньше: пользователь загружает список ТЕСТОВЫХ точек (ячейка 2.1), а 
контрольные точки подбираются из ВСЕЙ базы точек (df_pos_list), кроме уже использованных.

Режим "Подбор по списку ТТ" - новый: пользователь загружает ПОЛНЫЙ список точек, среди которых разрешено 
искать пары (в этом режиме ячейка 2.1 используется для загрузки этого списка, а не списка тестовых точек). 
Алгоритм сам определяет какие точки из этого списка станут тестовыми, а какие контрольными, и ищет пары 
ТОЛЬКО внутри этого списка (подробности в ячейке 3.5). Одна и та же точка не может быть использована дважды - 
ни как тестовая, ни как контрольная для другой тестовой точки.

ИСПРАВЛЕНО: изначально здесь использовался RadioButtons с CSS-трюком (flex-row) для горизонтального 
расположения - на практике это не сработало, кнопки всё равно выстроились вертикально (ipywidgets не 
гарантирует такое поведение для RadioButtons, это неофициальный обходной путь). Заменил на ToggleButtons - 
у этого виджета горизонтальное расположение работает "из коробки", без дополнительных настроек, а ведёт себя 
он точно так же, как RadioButtons: выбрать можно только один из двух вариантов, значение читается через 
mode_selector.value.
'''

mode_selector = widgets.ToggleButtons(
    options=['Подбор по всей АКБ', 'Подбор по списку ТТ'],
    value='Подбор по всей АКБ',
    description='Режим работы:',
    style={'description_width': 'initial'},
)

display(mode_selector)

In [ ]:
'''
Ячейка номер: 2.1. Создаём виджет, который будет загружать Excel файл со списком точек.

Смысл загружаемого списка зависит от режима, выбранного в ячейке 2.0:
- В режиме "Подбор по всей АКБ" - это список ТЕСТОВЫХ точек, для которых нужно подобрать контрольные.
- В режиме "Подбор по списку ТТ" - это ПОЛНЫЙ список точек, внутри которого нужно искать и тестовые, и 
  контрольные точки (см. ячейку 3.5).

Переменная, в которую загружается список, теперь называется uploaded_pos_list (раньше называлась 
test_pos_list) - имя изменено, чтобы не вводить в заблуждение в режиме "Подбор по списку ТТ", где это 
не список тестовых точек. Под кнопкой загрузки добавлена подсказка, которая обновляется при смене режима.
'''

uploaded_pos_list = []

# Создаём виджет загрузки (только .xlsx файлы)
uploader = widgets.FileUpload(accept='.xlsx',
                              multiple=False,
                              description='Загрузка списка точек',
                              layout=widgets.Layout(width='20%', height='40px'),
                              button_style='warning'                              
                              )
output_file = widgets.Output()
caption_upload = widgets.Output()

def redraw_upload_caption():
    with caption_upload:
        clear_output(wait=True)
        if mode_selector.value == 'Подбор по всей АКБ':
            print('Загрузите список ТЕСТОВЫХ точек (столбец SHIP_TO)')
        else:
            print('Загрузите ПОЛНЫЙ список точек, внутри которого нужно искать пары (столбец SHIP_TO)')

def on_upload(change):
    global uploaded_pos_list
    if uploader.value:
        # Получаем первый загруженный файл
        uploaded_file = uploader.value[0]
        content = uploaded_file['content']
        print(content)
        # Читаем Excel из байтов
        uploaded_pos_list = pd.read_excel(io.BytesIO(content))
        with output_file:
            clear_output()
            print("Загружено строк: " + str(uploaded_pos_list.shape[0]) )
            #display(uploaded_pos_list.head(5))  # Показываем первые строки

uploader.observe(on_upload, names='value')
mode_selector.observe(lambda change: redraw_upload_caption(), names='value')
redraw_upload_caption()
display(widgets.VBox([caption_upload, uploader, output_file]))

-----

In [ ]:
'''
Ячейка номер: 2.2. Создаем виджет, который позволит пользователю выбрать по каким конкретно признакам из иерархии торговых точек нужно осуществить подбор схожих контрольных торговых точек
'''


# Удобный выбор территориальных признаков
feature_list_geo = []
options_geo = ['RSD', 'BUSINESS_UNIT', 'SALES_GROUP', 'CLIENT', 'CUSTOMER_GROUP', 'CITY']

# Создаем чекбоксы
checkboxes_geo = [widgets.Checkbox(value=False, description=option_geo) for option_geo in options_geo]

# Блок вывода
output_feature = widgets.Output()

# Функция обновления списка выбранных фич
def on_change_feature(change):
    global feature_list_geo
    selected_geo = sorted([
        checkbox_geo.description
        for checkbox_geo in checkboxes_geo
        if checkbox_geo.value
    ])
    feature_list_geo = selected_geo

    with output_feature:
        output_feature.clear_output()
        print(f'Для поиска пар будут использоваться только точки с одинаковыми: {feature_list_geo}')

# Подписываем каждый чекбокс на изменение
for checkbox_geo in checkboxes_geo:
    checkbox_geo.observe(on_change_feature, names='value')

# Отображаем элементы
wig = widgets.VBox([
    widgets.GridBox(
        checkboxes_geo,
        layout=widgets.Layout(grid_template_columns="repeat(2, 250px)")
    )
])

display(wig, output_feature)

-----

In [ ]:
'''
Ячейка номер: 2.3. Создаём виджет для выбора временного периода, за который будут рассчитываться KPI.

Как это работает:
1. Пользователь двигает два ползунка (виджет IntRangeSlider) - левый задаёт начало периода, правый - конец.
2. Под слайдером есть поле TagsInput, куда можно вручную ввести номера отдельных недель в формате "202543"
   (год + номер недели), которые нужно ИСКЛЮЧИТЬ из расчёта, даже если они попадают в выбранный период
   (например, неделя с распродажей или другой аномалией).

Важный технический момент (почему слайдер сделан именно так, а не по самим номерам недель напрямую):
между неделей 202552 (последняя неделя 2025 года) и неделей 202601 (первая неделя 2026 года) есть "разрыв" -
чисел 202553, 202554 ... 202600 не существует. Если бы слайдер двигался прямо по номерам недель, то на
границе годов на шкале образовалась бы "мёртвая зона" длиной почти в 50 недель, по которой ползунок двигался
бы, а набор доступных недель не менялся бы - это неудобно и запутывает пользователя.

Поэтому слайдер двигается не по номерам недель, а по порядковым индексам (0, 1, 2 ... 103) в заранее
подготовленном списке all_weeks, где недели идут подряд без разрывов (202501 ... 202552, затем 202601 ... 202652).
После того как пользователь выбрал диапазон индексов, мы переводим их обратно в реальные номера недель.
Так шкала слайдера остаётся ровной, без "дыр", а результат (week_list) содержит только реально существующие недели.
'''

# Формируем полный список допустимых недель по порядку: 202501 ... 202552, затем 202601 ... 202652
weeks_2025 = [f'2025{week:02d}' for week in range(1, 53)]
weeks_2026 = [f'2026{week:02d}' for week in range(1, 53)]
all_weeks = weeks_2025 + weeks_2026

week_list = all_weeks.copy()  # Глобальная переменная со списком недель, которые попадут в расчёт KPI

# Слайдер диапазона периода (двигается по индексам списка all_weeks, а не по самим номерам недель - см. пояснение выше)
sl_week_range = widgets.IntRangeSlider(
    value=[0, len(all_weeks) - 1],
    min=0,
    max=len(all_weeks) - 1,
    step=1,
    description='Период:',
    continuous_update=False,
    layout=widgets.Layout(width='500px')
)

# Поле для ручного ввода недель, которые нужно исключить из расчёта (даже если они попадают в выбранный период)
tags_exclude_weeks = widgets.TagsInput(
    value=[],
    allowed_tags=all_weeks,   # разрешаем вводить только реально существующие номера недель
    allow_duplicates=False
)

label_period = widgets.Label('Период для расчёта KPI:')
label_exclude = widgets.Label('Исключить отдельные недели (введите номер вида 202543 и нажмите Enter):')
output_week = widgets.Output()

def on_week_change(change):
    global week_list
    start_idx, end_idx = sl_week_range.value
    selected_weeks = all_weeks[start_idx : end_idx + 1]
    excluded_weeks = list(tags_exclude_weeks.value)
    week_list = [week for week in selected_weeks if week not in excluded_weeks]

    with output_week:
        clear_output(wait=True)
        print(f'Выбранный период: с {selected_weeks[0]} по {selected_weeks[-1]}')
        if excluded_weeks:
            print(f'Исключённые недели: {excluded_weeks}')
        print(f'Итого недель, которые попадут в расчёт: {len(week_list)}')

sl_week_range.observe(on_week_change, names='value')
tags_exclude_weeks.observe(on_week_change, names='value')

display(widgets.VBox([label_period, sl_week_range, label_exclude, tags_exclude_weeks, output_week]))
on_week_change(None)  # первичная отрисовка с настройками по умолчанию (весь период целиком)

-----

In [ ]:
'''
Ячейка номер: 2.4. Создаем виджет, который позволит пользователю выбрать параметры (KPI) по которым нужно будет подобрать торговые точки. Для каждого выбранного параметра (KPI) 
пользователь может настроить диапазон допустимых отклонений. Если торговая точка укладывается в диапазон, то она может быть выбрана контрольной точкой, если не укладывается, то 
не может.

Что изменилось:
1. Исправлена вёрстка: подписи KPI ('FACING', 'SHELF SHARE' и т.д.) обрезались и выглядели как 'FACIN...'. 
   Причина - у виджета Checkbox есть собственная внутренняя ширина под текст подписи, которая по умолчанию не 
   зависит от layout.width. Добавлен параметр style={'description_width': 'initial'} (он говорит виджету 
   "не обрезай подпись, покажи её целиком") и немного расширен layout.
2. Для KPI 'SALES TREND' и 'MS TREND' допуск задаётся теперь НЕ множителем (0.5x - 1.5x), а максимально 
   допустимой разницей между углами наклона тренда тестовой и контрольной точки, в градусах.
   Как это работает: наклон тренда (slope, единицы - kCAF/неделя или доля рынка/неделя) переводится в угол 
   через arctan (это делается в ячейке 3.1). Дальше при подборе контрольных точек (ячейка 3.4) для каждого 
   кандидата считается разница углов с тестовой точкой ПО ТОМУ ЖЕ САМОМУ KPI (SALES TREND сравнивается только 
   с SALES TREND, MS TREND - только с MS TREND, между собой эти два KPI не сравниваются) - и кандидат подходит, 
   если эта разница не превышает заданный на слайдере допуск.
   Для остальных KPI (kCAF, OSA, VISITS, PICOS, FACING, SHELF SHARE, MARKET SHARE) допуск остаётся прежним - 
   множитель от значения тестовой точки.
'''

options_kpi = ['kCAF', 'OSA', 'VISITS', 'PICOS', 'FACING', 'SHELF SHARE', 'MARKET SHARE', 'SALES TREND', 'MS TREND']
options_kpi_angle = ['SALES TREND', 'MS TREND']  # эти KPI используют допуск в градусах, а не множитель

kpi_list = []
kpi_ranges = {}
checkboxes = {}
sliders = {}
rows = {}

output = widgets.Output()

def sync_globals():
    global kpi_list, kpi_ranges
    kpi_list = [name for name in options_kpi if checkboxes[name].value]
    kpi_ranges = {name: sliders[name].value for name in kpi_list}
    for name in kpi_list:
        globals()[f'kpi_{name}'] = sliders[name].value

def redraw_output():
    with output:
        clear_output(wait=True)
        print(f"Выбранные KPI: {kpi_list}")
        #print("Диапазоны KPI:")
        #for name in kpi_list:
        #    print(f"{name}: {sliders[name].value}")

def on_checkbox_change(change):
    kpi_name = change.owner.description
    if change.new:
        sliders[kpi_name].layout.display = 'flex'
    else:
        sliders[kpi_name].layout.display = 'none'
        if f'kpi_{kpi_name}' in globals():
            globals().pop(f'kpi_{kpi_name}', None)
    sync_globals()
    redraw_output()

def on_slider_change(change):
    kpi_name = change.owner.description
    globals()[f'kpi_{kpi_name}'] = change.new
    sync_globals()
    redraw_output()

for name in options_kpi:
    cb = widgets.Checkbox(
        value=False,
        description=name,
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='180px')
    )

    if name in options_kpi_angle:
        # Для трендов - допуск как максимально допустимая разница углов, в градусах (0-90)
        sl = widgets.FloatSlider(
            value=10,
            min=0,
            max=90,
            step=1,
            description='Допуск, °',
            continuous_update=False,
            layout=widgets.Layout(width='600px', display='none')
        )
    else:
        # Для остальных KPI - допуск как множитель от значения тестовой точки (как и раньше)
        sl = widgets.FloatRangeSlider(
            value=(0.9, 1.1),
            min=0.0,
            max=3.0,
            step=0.05,
            description='Откл. %',
            continuous_update=False,
            layout=widgets.Layout(width='600px', display='none')
        )

    cb.observe(on_checkbox_change, names='value')
    sl.observe(on_slider_change, names='value')

    checkboxes[name] = cb
    sliders[name] = sl
    rows[name] = widgets.HBox([cb, sl])

ui = widgets.VBox([rows[name] for name in options_kpi])

display(ui, output)

sync_globals()
redraw_output()

-----

3. БЛОК ЛОГИКИ ДЛЯ ПОИСКА КОНТРОЛЬНЫХ ТОЧЕК

In [ ]:
'''
Ячейка номер: 3.1. Создаём функцию которая принимает на входе все датафреймы с данными по расчёту KPI, убирает из них 
недели, которые пользователь не выбрал (week_list из ячейки 2.3), сворачивает данные до одного значения на неделю, 
удаляет выбросы (IQR), и превращает недельные значения KPI в одно значение за весь период для каждой торговой точки.

Функция считает только те KPI, которые пользователь реально отметил галочкой в ячейке 2.4 (kpi_list), и возвращает 
результат словарём (dict) - к каждому датафрейму можно обратиться по понятному ключу, например cut_results['OSA'].

УСКОРЕНО: раньше cut_kpi_mean и cut_kpi_trend считали каждую точку отдельно в Python-цикле (for ship_to, group in 
df.groupby(...)) - при большом числе точек (тысячи) это заметно медленно, т.к. на каждой итерации вызывались 
.quantile() и np.polyfit() по отдельности. Переписано на полностью векторизованный подход - без единого 
Python-цикла по точкам:
- Квартили (Q1, Q3) для ВСЕХ точек сразу считаются одним групповым вызовом grouped.quantile(0.25) - у pandas 
  это быстрый встроенный (не Python-цикл "под капотом") способ посчитать квантиль по каждой группе.
- Наклон линейного тренда (slope) считается не через np.polyfit по каждой точке отдельно, а через явную 
  формулу линейной регрессии (slope = Sxy/Sxx), где все нужные суммы (sum_x, sum_y, sum_xy, sum_xx) тоже 
  считаются одним групповым вызовом .agg(...) для всех точек сразу.
На тесте с 3000 точками и 104 неделями (312 тысяч строк) это ускорило расчёт примерно в 20 раз (с ~4.5 до 
~0.2 секунды на каждую функцию), при этом результаты полностью совпадают со старым способом (проверено).
'''

def get_weekly_series(df_raw, se_week_list, value_col):
    '''Фильтрует сырые данные по выбранному периоду (se_week_list) и сворачивает их до одного значения 
    на неделю для каждой точки (среднее) - на случай нескольких записей за одну неделю.
    Возвращает датафрейм с колонками SHIP_TO, YEAR_WEEK, value_col.'''
    df_filtered = df_raw.merge(se_week_list, on='YEAR_WEEK', how='inner')
    df_weekly = df_filtered.groupby(['SHIP_TO', 'YEAR_WEEK'])[value_col].mean().reset_index()
    return df_weekly

def remove_outliers_iqr(series):
    '''Убирает выбросы из числового ряда ОДНОЙ точки методом IQR. Используется в разделе 5 (графики), 
    где обрабатывается всего одна точка за раз - там скорость не критична, поэтому оставлена простая версия.
    Если в ряду 3 и менее значений - выбросы не убираем (граница IQR на маленькой выборке ненадёжна).'''
    if len(series) <= 3:
        return series
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    return series[(series >= lower_bound) & (series <= upper_bound)]

def get_clean_weekly_series(df_raw, se_week_list, value_col):
    '''Векторизованная версия очистки от выбросов - обрабатывает СРАЗУ ВСЕ точки, без Python-цикла.
    Сворачивает данные до одного значения на неделю, считает границы IQR для каждой точки одним групповым 
    вызовом, и возвращает только "чистые" (без выбросов) строки, отсортированные по неделе внутри точки.'''
    df_weekly = get_weekly_series(df_raw, se_week_list, value_col)
    df_weekly = df_weekly.sort_values(['SHIP_TO', 'YEAR_WEEK'])

    grouped = df_weekly.groupby('SHIP_TO')[value_col]
    stats = pd.DataFrame({
        'q1': grouped.quantile(0.25),
        'q3': grouped.quantile(0.75),
        'n': grouped.size(),
    }).reset_index()
    stats['iqr'] = stats['q3'] - stats['q1']
    stats['lower'] = stats['q1'] - 1.5 * stats['iqr']
    stats['upper'] = stats['q3'] + 1.5 * stats['iqr']
    # Если в группе 3 и менее значений - выбросы не убираем (расширяем границы до +-бесконечности)
    stats.loc[stats['n'] <= 3, 'lower'] = -np.inf
    stats.loc[stats['n'] <= 3, 'upper'] = np.inf

    df_merged = df_weekly.merge(stats[['SHIP_TO', 'lower', 'upper']], on='SHIP_TO', how='left')
    is_outlier = (df_merged[value_col] < df_merged['lower']) | (df_merged[value_col] > df_merged['upper'])
    df_clean = df_merged.loc[~is_outlier].drop(columns=['lower', 'upper'])
    return df_clean

def cut_kpi_mean(df_raw, se_week_list, value_col, output_col, cast_type):
    '''Считает среднее значение показателя за период для каждой точки, с очисткой недельного ряда от 
    выбросов (IQR). Используется для kCAF, OSA, VISITS, PICOS, FACING, SHELF SHARE, MARKET SHARE.'''
    df_clean = get_clean_weekly_series(df_raw, se_week_list, value_col)
    result = df_clean.groupby('SHIP_TO')[value_col].mean().reset_index()
    result.rename(columns={value_col: output_col}, inplace=True)
    result[output_col] = result[output_col].astype(cast_type)
    return result

def cut_kpi_trend(df_raw, se_week_list, value_col, slope_col, angle_col):
    '''Считает наклон линейного тренда (slope) и угол наклона в градусах для показателя за период, с 
    очисткой недельного ряда от выбросов (IQR). Используется для SALES TREND и MS TREND.

    Наклон считается по формуле линейной регрессии slope = Sxy / Sxx, где x - порядковый номер недели внутри 
    очищенного (без выбросов) ряда точки (0, 1, 2, ...), а Sxy и Sxx - суммы отклонений от среднего. 
    Это математически то же самое, что и np.polyfit(x, y, 1)[0], но считается сразу для всех точек одним 
    групповым вызовом, без Python-цикла по каждой точке отдельно.'''
    df_clean = get_clean_weekly_series(df_raw, se_week_list, value_col)
    df_clean = df_clean.sort_values(['SHIP_TO', 'YEAR_WEEK'])

    # x - порядковый номер недели внутри очищенного ряда каждой точки (0, 1, 2, ...)
    df_clean['x'] = df_clean.groupby('SHIP_TO').cumcount()
    df_clean['xy'] = df_clean['x'] * df_clean[value_col]
    df_clean['xx'] = df_clean['x'] ** 2

    agg = df_clean.groupby('SHIP_TO').agg(
        n=(value_col, 'size'),
        sum_x=('x', 'sum'),
        sum_y=(value_col, 'sum'),
        sum_xy=('xy', 'sum'),
        sum_xx=('xx', 'sum'),
    ).reset_index()

    agg['Sxy'] = agg['sum_xy'] - agg['sum_x'] * agg['sum_y'] / agg['n']
    agg['Sxx'] = agg['sum_xx'] - agg['sum_x'] ** 2 / agg['n']
    agg[slope_col] = agg['Sxy'] / agg['Sxx']
    agg.loc[agg['n'] < 2, slope_col] = 0  # меньше 2 точек - тренд посчитать нельзя
    agg[angle_col] = np.degrees(np.arctan(agg[slope_col]))

    return agg[['SHIP_TO', slope_col, angle_col]]

def cut_dataframes(week_list, df_sales, df_osa, df_visits, df_facing_fact, df_picos, df_shelfshare, df_marketshare):

    se_week_list = pd.Series(map(int, week_list), name='YEAR_WEEK')
    cut_results = {}

    if 'kCAF' in kpi_list:
        cut_results['SALES'] = cut_kpi_mean(df_sales, se_week_list, 'kCAF', 'kCAF', 'int')

    if 'OSA' in kpi_list:
        cut_results['OSA'] = cut_kpi_mean(df_osa, se_week_list, 'OSA', 'OSA', 'float')

    if 'VISITS' in kpi_list:
        cut_results['VISITS'] = cut_kpi_mean(df_visits, se_week_list, 'PHOTO_AUDITS_PER_WEEK', 'PHOTO_AUDITS_PER_WEEK', 'float')

    if 'FACING' in kpi_list:
        cut_results['FACING'] = cut_kpi_mean(df_facing_fact, se_week_list, 'GROUP_FACT', 'GROUP_FACT', 'int')

    if 'PICOS' in kpi_list:
        cut_results['PICOS'] = cut_kpi_mean(df_picos, se_week_list, 'PICOS_SCORE_FACT', 'PICOS_SCORE_FACT', 'int')

    if 'SHELF SHARE' in kpi_list:
        cut_results['SHELF_SHARE'] = cut_kpi_mean(df_shelfshare, se_week_list, 'SHELF_SHARE', 'SHELF_SHARE_AVG', 'float')

    if 'MARKET SHARE' in kpi_list:
        cut_results['MARKET_SHARE'] = cut_kpi_mean(df_marketshare, se_week_list, 'MARKET_SHARE', 'MARKET_SHARE_AVG', 'float')

    if 'SALES TREND' in kpi_list:
        cut_results['SALES_TREND'] = cut_kpi_trend(df_sales, se_week_list, 'kCAF', 'SALES_SLOPE', 'SALES_TREND_ANGLE')

    if 'MS TREND' in kpi_list:
        cut_results['MS_TREND'] = cut_kpi_trend(df_marketshare, se_week_list, 'MARKET_SHARE', 'MS_SLOPE', 'MS_TREND_ANGLE')

    return cut_results

In [ ]:
'''
Ячейка номер: 3.2. Создаём функцию которая принимает на входе словарь с рассчитанными KPI (из ячейки 3.1) и добавляет их 
к полному списку торговых точек в системе. Также сохраняется статус добавления: если данные по какой-либо точке 
отсутствуют, пользователь сможет это увидеть в колонке DATA_IN_...

Обновлено: функция теперь принимает cut_results (словарь из ячейки 3.1) вместо пяти отдельных датафреймов, 
и добавлена обработка новых KPI: SHELF SHARE, MARKET SHARE, SALES TREND, MS TREND.
'''

def calc_kpi_for_test_pos(df_pos_list, cut_results):

    df_pos_list_with_kpi = pd.DataFrame(df_pos_list)

    if 'OSA' in kpi_list: 
        df_pos_list_with_kpi = df_pos_list_with_kpi.merge(cut_results['OSA'], on='SHIP_TO', how='left', indicator=True )
        df_pos_list_with_kpi['_merge'] = df_pos_list_with_kpi['_merge'].astype('string')
        df_pos_list_with_kpi.rename(columns={'_merge': 'DATA_IN_OSA' }, inplace=True)
        df_pos_list_with_kpi.replace({'DATA_IN_OSA' : { 'both' : 'Ok', 'left_only' : 'No_OSA_data'}}, inplace=True)

    if 'kCAF' in kpi_list: 
        df_pos_list_with_kpi = df_pos_list_with_kpi.merge(cut_results['SALES'], on='SHIP_TO', how='left', indicator=True )
        df_pos_list_with_kpi['_merge'] = df_pos_list_with_kpi['_merge'].astype('string')
        df_pos_list_with_kpi.rename(columns={'_merge': 'DATA_IN_SALES' }, inplace=True)
        df_pos_list_with_kpi.replace({'DATA_IN_SALES' : { 'both' : 'Ok', 'left_only' : 'No_SALES_data'}}, inplace=True)

    if 'VISITS' in kpi_list: 
        df_pos_list_with_kpi = df_pos_list_with_kpi.merge(cut_results['VISITS'], on='SHIP_TO', how='left', indicator=True )
        df_pos_list_with_kpi['_merge'] = df_pos_list_with_kpi['_merge'].astype('string')
        df_pos_list_with_kpi.rename(columns={'_merge': 'DATA_IN_VISITS' }, inplace=True)
        df_pos_list_with_kpi.replace({'DATA_IN_VISITS' : { 'both' : 'Ok', 'left_only' : 'No_VISITS_data'}}, inplace=True)
    
    if 'FACING' in kpi_list: 
        df_pos_list_with_kpi = df_pos_list_with_kpi.merge(cut_results['FACING'], on='SHIP_TO', how='left', indicator=True )
        df_pos_list_with_kpi['_merge'] = df_pos_list_with_kpi['_merge'].astype('string')
        df_pos_list_with_kpi.rename(columns={'_merge': 'DATA_IN_FACING' }, inplace=True)
        df_pos_list_with_kpi.replace({'DATA_IN_FACING' : { 'both' : 'Ok', 'left_only' : 'No_FACING_data'}}, inplace=True)

    if 'PICOS' in kpi_list: 
        df_pos_list_with_kpi = df_pos_list_with_kpi.merge(cut_results['PICOS'], on='SHIP_TO', how='left', indicator=True )
        df_pos_list_with_kpi['_merge'] = df_pos_list_with_kpi['_merge'].astype('string')
        df_pos_list_with_kpi.rename(columns={'_merge': 'DATA_IN_PICOS' }, inplace=True)
        df_pos_list_with_kpi.replace({'DATA_IN_PICOS' : { 'both' : 'Ok', 'left_only' : 'No_PICOS_data'}}, inplace=True)

    if 'SHELF SHARE' in kpi_list:
        df_pos_list_with_kpi = df_pos_list_with_kpi.merge(cut_results['SHELF_SHARE'], on='SHIP_TO', how='left', indicator=True )
        df_pos_list_with_kpi['_merge'] = df_pos_list_with_kpi['_merge'].astype('string')
        df_pos_list_with_kpi.rename(columns={'_merge': 'DATA_IN_SHELF_SHARE' }, inplace=True)
        df_pos_list_with_kpi.replace({'DATA_IN_SHELF_SHARE' : { 'both' : 'Ok', 'left_only' : 'No_SHELF_SHARE_data'}}, inplace=True)

    if 'MARKET SHARE' in kpi_list:
        df_pos_list_with_kpi = df_pos_list_with_kpi.merge(cut_results['MARKET_SHARE'], on='SHIP_TO', how='left', indicator=True )
        df_pos_list_with_kpi['_merge'] = df_pos_list_with_kpi['_merge'].astype('string')
        df_pos_list_with_kpi.rename(columns={'_merge': 'DATA_IN_MARKET_SHARE' }, inplace=True)
        df_pos_list_with_kpi.replace({'DATA_IN_MARKET_SHARE' : { 'both' : 'Ok', 'left_only' : 'No_MARKET_SHARE_data'}}, inplace=True)

    if 'SALES TREND' in kpi_list:
        df_pos_list_with_kpi = df_pos_list_with_kpi.merge(cut_results['SALES_TREND'], on='SHIP_TO', how='left', indicator=True )
        df_pos_list_with_kpi['_merge'] = df_pos_list_with_kpi['_merge'].astype('string')
        df_pos_list_with_kpi.rename(columns={'_merge': 'DATA_IN_SALES_TREND' }, inplace=True)
        df_pos_list_with_kpi.replace({'DATA_IN_SALES_TREND' : { 'both' : 'Ok', 'left_only' : 'No_SALES_TREND_data'}}, inplace=True)

    if 'MS TREND' in kpi_list:
        df_pos_list_with_kpi = df_pos_list_with_kpi.merge(cut_results['MS_TREND'], on='SHIP_TO', how='left', indicator=True )
        df_pos_list_with_kpi['_merge'] = df_pos_list_with_kpi['_merge'].astype('string')
        df_pos_list_with_kpi.rename(columns={'_merge': 'DATA_IN_MS_TREND' }, inplace=True)
        df_pos_list_with_kpi.replace({'DATA_IN_MS_TREND' : { 'both' : 'Ok', 'left_only' : 'No_MS_TREND_data'}}, inplace=True)

    df_pos_list_with_kpi['PAIR_NUMBER'] = 0
    df_pos_list_with_kpi['POS_TYPE'] = '-'
    return df_pos_list_with_kpi

In [ ]:
'''
Ячейка номер: 3.3. Создаём виджет, который позволит пользователю выбрать сколько схожих контрольных точек нужно подобрать для каждой тестовой точки от 1 до 10
'''

print("Кол-во контрольных ТТ:")
sl_number_control_pos = widgets.IntSlider(
    value=1,
    min=1,
    max=10,
    step=1,
    description='Кол-во ТТ:',
    disabled=False,
    continuous_update=True,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)

sl_number_control_pos

In [ ]:
'''
Ячейка номер: 3.4. Создаём функцию по поиску контрольных точек для каждой тестовой точки (режим "Подбор по всей АКБ").

Обновлено:
1. Логика фильтрации кандидатов по территориальным признакам (ячейка 2.2) и допускам по KPI (ячейка 2.4) 
   вынесена в отдельную функцию filter_candidates_by_criteria - она же используется в ячейке 3.5 для нового 
   режима "Подбор по списку ТТ", чтобы не дублировать код и логика подбора не разъехалась между режимами 
   при будущих правках.
2. Добавлен необязательный параметр progress_bar - если он передан, функция обновляет его значение на 
   каждой итерации цикла, чтобы пользователь видел индикатор прогресса расчёта (ячейка 4.1).
'''

def filter_candidates_by_criteria(df_candidates, df_one_test_pos):
    '''Сужает список кандидатов в контрольные точки по территориальным признакам (ячейка 2.2) и по 
    допустимым отклонениям KPI (ячейка 2.4). Используется и в режиме "Подбор по всей АКБ" (эта ячейка), 
    и в режиме "Подбор по списку ТТ" (ячейка 3.5) - логика подбора одна и та же, отличается только то, 
    среди каких точек ведётся поиск.'''

    df_result = df_candidates

    if 'RSD' in feature_list_geo:
        df_result = df_result.loc[df_result['RSD'] == df_one_test_pos['RSD'][0]]
    if 'BUSINESS_UNIT' in feature_list_geo:
        df_result = df_result.loc[df_result['BUSINESS_UNIT'] == df_one_test_pos['BUSINESS_UNIT'][0]]
    if 'SALES_GROUP' in feature_list_geo:
        df_result = df_result.loc[df_result['SALES_GROUP'] == df_one_test_pos['SALES_GROUP'][0]]
    if 'CITY' in feature_list_geo:
        df_result = df_result.loc[df_result['CITY'] == df_one_test_pos['CITY'][0]]
    if 'CLIENT' in feature_list_geo:
        df_result = df_result.loc[df_result['CLIENT'] == df_one_test_pos['CLIENT'][0]]
    if 'CUSTOMER_GROUP' in feature_list_geo:
        df_result = df_result.loc[df_result['CUSTOMER_GROUP'] == df_one_test_pos['CUSTOMER_GROUP'][0]]

    if 'OSA' in kpi_list:
        df_result = df_result.loc[df_result['OSA'] >= df_one_test_pos['OSA'][0] * kpi_ranges['OSA'][0]]
        df_result = df_result.loc[df_result['OSA'] <= df_one_test_pos['OSA'][0] * kpi_ranges['OSA'][1]]
    if 'kCAF' in kpi_list:
        df_result = df_result.loc[df_result['kCAF'] >= df_one_test_pos['kCAF'][0] * kpi_ranges['kCAF'][0]]
        df_result = df_result.loc[df_result['kCAF'] <= df_one_test_pos['kCAF'][0] * kpi_ranges['kCAF'][1]]
    if 'VISITS' in kpi_list:
        df_result = df_result.loc[df_result['PHOTO_AUDITS_PER_WEEK'] >= df_one_test_pos['PHOTO_AUDITS_PER_WEEK'][0] * kpi_ranges['VISITS'][0]]
        df_result = df_result.loc[df_result['PHOTO_AUDITS_PER_WEEK'] <= df_one_test_pos['PHOTO_AUDITS_PER_WEEK'][0] * kpi_ranges['VISITS'][1]]
    if 'PICOS' in kpi_list:
        df_result = df_result.loc[df_result['PICOS_SCORE_FACT'] >= df_one_test_pos['PICOS_SCORE_FACT'][0] * kpi_ranges['PICOS'][0]]
        df_result = df_result.loc[df_result['PICOS_SCORE_FACT'] <= df_one_test_pos['PICOS_SCORE_FACT'][0] * kpi_ranges['PICOS'][1]]
    if 'FACING' in kpi_list:
        df_result = df_result.loc[df_result['GROUP_FACT'] >= df_one_test_pos['GROUP_FACT'][0] * kpi_ranges['FACING'][0]]
        df_result = df_result.loc[df_result['GROUP_FACT'] <= df_one_test_pos['GROUP_FACT'][0] * kpi_ranges['FACING'][1]]
    if 'SHELF SHARE' in kpi_list:
        df_result = df_result.loc[df_result['SHELF_SHARE_AVG'] >= df_one_test_pos['SHELF_SHARE_AVG'][0] * kpi_ranges['SHELF SHARE'][0]]
        df_result = df_result.loc[df_result['SHELF_SHARE_AVG'] <= df_one_test_pos['SHELF_SHARE_AVG'][0] * kpi_ranges['SHELF SHARE'][1]]
    if 'MARKET SHARE' in kpi_list:
        df_result = df_result.loc[df_result['MARKET_SHARE_AVG'] >= df_one_test_pos['MARKET_SHARE_AVG'][0] * kpi_ranges['MARKET SHARE'][0]]
        df_result = df_result.loc[df_result['MARKET_SHARE_AVG'] <= df_one_test_pos['MARKET_SHARE_AVG'][0] * kpi_ranges['MARKET SHARE'][1]]
    if 'SALES TREND' in kpi_list:
        test_angle_sales = df_one_test_pos['SALES_TREND_ANGLE'][0]
        angle_diff_sales = (df_result['SALES_TREND_ANGLE'] - test_angle_sales).abs()
        df_result = df_result.loc[angle_diff_sales <= kpi_ranges['SALES TREND']]
    if 'MS TREND' in kpi_list:
        test_angle_ms = df_one_test_pos['MS_TREND_ANGLE'][0]
        angle_diff_ms = (df_result['MS_TREND_ANGLE'] - test_angle_ms).abs()
        df_result = df_result.loc[angle_diff_ms <= kpi_ranges['MS TREND']]

    return df_result


global df_pair_list
df_pair_list = pd.DataFrame([])


def find_pairs(uploaded_pos_list, df_pos_list_with_kpi, progress_bar=None):

    global df_pair_list
    df_pair_list = pd.DataFrame([])

    total_test_points = uploaded_pos_list.shape[0]

    for index, df_row in uploaded_pos_list.iterrows():
        #Сохраняем данные по тестовой точке
        
        df_one_test_pos = df_pos_list_with_kpi.loc[df_pos_list_with_kpi['SHIP_TO'] ==  df_row['SHIP_TO']].reset_index()
        if df_one_test_pos.shape[0] == 0:
            df_one_test_pos.loc[0, ['SHIP_TO', 'RSD' ]] = [df_row['SHIP_TO'], 'POS not found in Optimum']
        df_one_test_pos['PAIR_NUMBER'] = index + 1
        df_one_test_pos['POS_TYPE'] = 'Test_POS'
        
        df_pair_list = pd.concat([df_pair_list, df_one_test_pos])
        df_pair_list = df_pair_list.reset_index().drop(columns=['index', 'level_0'])
        df_pair_list['SHIP_TO'] = df_pair_list['SHIP_TO'].astype('int')
        df_pair_list['PAIR_NUMBER'] = df_pair_list['PAIR_NUMBER'].astype('int')   
    
    
        ## Подбираем контрольную точку по территории и KPI
    
        df_used_pos = pd.DataFrame(pd.Series(pd.concat([pd.DataFrame(df_pair_list['SHIP_TO']), uploaded_pos_list])['SHIP_TO'].unique(), name='SHIP_TO'))
        df_one_control_pos =  df_pos_list_with_kpi.merge(df_used_pos, how='left', on='SHIP_TO', indicator=True)
        df_one_control_pos =  df_one_control_pos.loc[df_one_control_pos['_merge'] == 'left_only'].drop(columns='_merge')
    
        df_one_control_pos = filter_candidates_by_criteria(df_one_control_pos, df_one_test_pos)
        
        needed_count = sl_number_control_pos.value
        found_count = df_one_control_pos.shape[0]

        if found_count > 0: 
            df_one_control_pos = df_one_control_pos.head(needed_count)
            df_one_control_pos['PAIR_NUMBER'] = index + 1
            df_one_control_pos['POS_TYPE'] = 'Control_POS'
            df_pair_list = pd.concat([df_pair_list, df_one_control_pos])
        else: 
            df_pair_list.loc[(df_pair_list['SHIP_TO'] == df_row['SHIP_TO']) & (df_pair_list['POS_TYPE'] == 'Test_POS'), 'POS_TYPE' ] = 'Test_POS_wo_pair'

        missing_count = needed_count - found_count
        if found_count > 0 and missing_count > 0:
            df_missing_rows = pd.DataFrame([{'SHIP_TO': 0, 'PAIR_NUMBER': index + 1, 'POS_TYPE': 'Control_POS_not_found'} for i in range(missing_count)])
            df_pair_list = pd.concat([df_pair_list, df_missing_rows])

        if progress_bar is not None:
            progress_bar.value = 10 + 90 * (index + 1) / total_test_points

    df_pair_list = df_pair_list.reset_index(drop=True)
    return df_pair_list          

In [ ]:
'''
Ячейка номер: 3.5. Назначение: Функция подбора пар для режима "Подбор по списку ТТ".

В этом режиме пользователь загружает не список тестовых точек, а ПОЛНЫЙ список точек (uploaded_pos_list), 
внутри которого нужно самостоятельно определить, какие точки станут тестовыми, а какие - контрольными.

Логика:
1. Берём точки из загруженного списка по порядку (как они шли в файле).
2. Первая точка, которой ещё не присвоена группа, становится тестовой (POS_TYPE = 'Test_POS').
3. Среди ОСТАЛЬНЫХ точек ТОГО ЖЕ загруженного списка (за вычетом уже использованных) ищем контрольные - той 
   же функцией filter_candidates_by_criteria, что и в режиме "Подбор по всей АКБ" (ячейка 3.4).
4. Найденные контрольные точки помечаются POS_TYPE = 'Control_POS' и считаются использованными.
5. Переходим к следующей неиспользованной точке в списке и повторяем, пока список не закончится.
6. Точка, которая уже была использована как тестовая или как контрольная, повторно в подборе не участвует - 
   ни как тестовая, ни как контрольная для другой пары. Это гарантируется множеством used_ship_to_set - 
   любая точка добавляется туда сразу, как только получает какую-либо роль.

Если тестовая точка не найдена в системе (df_pos_list) или для неё не нашлось ни одной контрольной точки 
внутри списка - обрабатывается точно так же, как в режиме "Подбор по всей АКБ" ('POS not found in Optimum' 
и 'Test_POS_wo_pair' соответственно).
'''

def find_pairs_within_pool(uploaded_pos_list, df_pos_list_with_kpi, progress_bar=None):

    global df_pair_list
    df_pair_list = pd.DataFrame([])

    # Список SHIP_TO из загруженного файла в том порядке, в котором они там перечислены.
    # drop_duplicates - на случай, если в файле случайно попалась одна и та же точка дважды.
    pool_ship_to_list = uploaded_pos_list['SHIP_TO'].astype('int').drop_duplicates().tolist()
    total_pool_size = len(pool_ship_to_list)

    used_ship_to_set = set()  # сюда добавляем все точки, которые уже получили какую-либо группу (тест или контроль)
    pair_number = 0

    for ship_to in pool_ship_to_list:

        if ship_to in used_ship_to_set:
            continue  # эта точка уже была задействована в одной из предыдущих пар - пропускаем

        pair_number += 1
        used_ship_to_set.add(ship_to)

        ## Сохраняем данные по тестовой точке

        df_one_test_pos = df_pos_list_with_kpi.loc[df_pos_list_with_kpi['SHIP_TO'] == ship_to].reset_index(drop=True)
        if df_one_test_pos.shape[0] == 0:
            df_one_test_pos = pd.DataFrame([{'SHIP_TO': ship_to, 'RSD': 'POS not found in Optimum'}])
        df_one_test_pos['PAIR_NUMBER'] = pair_number
        df_one_test_pos['POS_TYPE'] = 'Test_POS'

        df_pair_list = pd.concat([df_pair_list, df_one_test_pos]).reset_index(drop=True)
        df_pair_list['SHIP_TO'] = df_pair_list['SHIP_TO'].astype('int')
        df_pair_list['PAIR_NUMBER'] = df_pair_list['PAIR_NUMBER'].astype('int')

        ## Подбираем контрольные точки, но ТОЛЬКО среди точек из загруженного списка, которые ещё не использованы

        df_pool_candidates = df_pos_list_with_kpi.loc[df_pos_list_with_kpi['SHIP_TO'].isin(pool_ship_to_list)]
        df_one_control_pos = df_pool_candidates.loc[~df_pool_candidates['SHIP_TO'].isin(used_ship_to_set)]

        df_one_control_pos = filter_candidates_by_criteria(df_one_control_pos, df_one_test_pos)

        needed_count = sl_number_control_pos.value
        found_count = df_one_control_pos.shape[0]

        if found_count > 0:
            df_one_control_pos = df_one_control_pos.head(needed_count)
            df_one_control_pos['PAIR_NUMBER'] = pair_number
            df_one_control_pos['POS_TYPE'] = 'Control_POS'
            df_pair_list = pd.concat([df_pair_list, df_one_control_pos]).reset_index(drop=True)
            used_ship_to_set.update(df_one_control_pos['SHIP_TO'].tolist())
        else:
            df_pair_list.loc[(df_pair_list['SHIP_TO'] == ship_to) & (df_pair_list['POS_TYPE'] == 'Test_POS'), 'POS_TYPE'] = 'Test_POS_wo_pair'

        missing_count = needed_count - found_count
        if found_count > 0 and missing_count > 0:
            df_missing_rows = pd.DataFrame([{'SHIP_TO': 0, 'PAIR_NUMBER': pair_number, 'POS_TYPE': 'Control_POS_not_found'} for i in range(missing_count)])
            df_pair_list = pd.concat([df_pair_list, df_missing_rows]).reset_index(drop=True)

        if progress_bar is not None and total_pool_size > 0:
            progress_bar.value = 10 + 90 * len(used_ship_to_set) / total_pool_size

    df_pair_list = df_pair_list.reset_index(drop=True)

    return df_pair_list

4. ПОДБОР ПАР И ВЫГРУЗКА РЕЗУЛЬТАТОВ

In [ ]:
'''
Ячейка номер: 4.1. Создаём виджет который после нажатия кнопки "Рассчитать" производит поиск контрольных точек для тестовых.

Обновлено:
1. Добавлен индикатор прогресса (виджет FloatProgress) справа от кнопки "Рассчитать" - показывает, на каком 
   этапе расчёта находится приложение, чтобы не создавалось ощущение, что интерфейс завис. Индикатор 
   обновляется в реальном времени по ходу расчёта: 0-5% - расчёт KPI по датафреймам (ячейка 3.1), 5-10% - 
   привязка KPI к точкам (ячейка 3.2), 10-100% - подбор пар (ячейка 3.4 или 3.5, в зависимости от режима), 
   с обновлением после обработки каждой точки. При ошибке индикатор становится красным (bar_style='danger'), 
   при успешном завершении - зелёным (bar_style='success').
   Обновления виджета видны пользователю сразу по ходу расчёта, а не только после его полного завершения - 
   именно поэтому это FloatProgress, а не просто текстовое сообщение "Идёт расчёт" в конце.
2. Расчёт теперь ветвится по режиму, выбранному в ячейке 2.0: 'Подбор по всей АКБ' использует find_pairs 
   (ячейка 3.4), 'Подбор по списку ТТ' использует find_pairs_within_pool (ячейка 3.5). В обоих случаях 
   в функцию передаётся uploaded_pos_list (загруженный в ячейке 2.1 файл) - его смысл (список тестовых точек 
   или полный список точек) зависит от выбранного режима.
'''

import traceback

def on_calculate_click(button):
    global df_pair_list
    global widget_project_start_date
    global widget_project_end_date
    global widget_project_name

    output_calc.clear_output()
    progress_bar.value = 0
    progress_bar.bar_style = 'info'

    with output_calc:
        try:
            cut_results = cut_dataframes(week_list, df_sales, df_osa, df_visits, df_facing_fact, df_picos, df_shelfshare, df_marketshare )
            progress_bar.value = 5

            df_pos_list_with_kpi = calc_kpi_for_test_pos(df_pos_list, cut_results )
            progress_bar.value = 10

            if mode_selector.value == 'Подбор по всей АКБ':
                df_pair_list = find_pairs(uploaded_pos_list, df_pos_list_with_kpi, progress_bar=progress_bar)
            else:
                df_pair_list = find_pairs_within_pool(uploaded_pos_list, df_pos_list_with_kpi, progress_bar=progress_bar)

            progress_bar.value = 100
            progress_bar.bar_style = 'success'

            print("Все функции успешно выполнены")
            print('Найдено пар: ' + str(df_pair_list[df_pair_list['POS_TYPE'] == 'Test_POS'].shape[0]))
            print('Точки без пары: ' + str(df_pair_list[(df_pair_list['RSD'] != 'POS not found in Optimum') & (df_pair_list['POS_TYPE'] == 'Test_POS_wo_pair')].shape[0]))
            print('Точек нет в SFA: ' + str(df_pair_list[df_pair_list['RSD'] == 'POS not found in Optimum'].shape[0]))
            print('Недостающих контрольных точек (не хватило до нужного количества): ' + str(df_pair_list[df_pair_list['POS_TYPE'] == 'Control_POS_not_found'].shape[0]))
            #display(df_pair_list)

            # Обновляем виджет раздела 5 (просмотр результатов), если он уже создан к этому моменту -
            # подтягиваем новый список пар и прячем прошлые графики, чтобы не запутать пользователя устаревшими данными
            if 'dropdown_pair' in globals():
                output_charts.clear_output()
                dropdown_pair.options = get_pair_options()
                dropdown_pair.value = None

        except Exception as e:
            progress_bar.bar_style = 'danger'
            print("Ошибка при расчёте, подробности ниже:")
            traceback.print_exc()

        # Переносим 'PAIR_NUMBER' и 'POS_TYPE' в начало датафрейма чтобы файл target_pos было проще импортировать в BI систему
    col_1 = 'PAIR_NUMBER'
    col_2 = 'POS_TYPE'
    target_pos = 1  # 10-я позиция слева, если считать с нуля
    cols = df_pair_list.columns.tolist()
    cols.remove(col_1)
    cols.remove(col_2)
    cols.insert(target_pos, col_2)
    cols.insert(target_pos, col_1)
    df_pair_list = df_pair_list[cols]


     # Добавляем столбцы с названием проекта и его датами в начало датафрейма
    project_start_date = widget_project_start_date.value
    project_end_date = widget_project_end_date.value
    project_start_date = "Дата не указана" if str(type(widget_project_start_date.value)) == "<class 'NoneType'>" else project_start_date.strftime("%d.%m.%Y")
    project_end_date = "Дата не указана" if str(type(widget_project_end_date.value)) == "<class 'NoneType'>" else project_end_date.strftime("%d.%m.%Y")
    project_name = 'Название не указано' if widget_project_name.value == '' else str(widget_project_name.value)
    df_pair_list.insert(loc=0, column='PROJECT_END_DATE', value=project_end_date)
    df_pair_list.insert(loc=0, column='PROJECT_START_DATE', value=project_start_date)
    df_pair_list.insert(loc=0, column='PROJECT_NAME', value=project_name)  

    df_pair_list.drop(columns=['ROUTE_NAME', 'AGENCY_NAME', 'ClientName', 'Proxi_Category_1', 'GMT', 'fid'], inplace=True) 

    return df_pair_list

calculate_button = widgets.Button(
    description='Рассчитать',
    layout=widgets.Layout(width='20%', height='40px'),
    button_style='danger',
    icon='check'
)

progress_bar = widgets.FloatProgress(
    value=0,
    min=0,
    max=100,
    description='Прогресс:',
    bar_style='info',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

output_calc = widgets.Output()

calculate_button.on_click(on_calculate_click)

display(widgets.VBox([widgets.HBox([calculate_button, progress_bar]), output_calc]))

------

In [ ]:
'''
Ячейка номер: 4.2. Создаём виджет который позволяет пользователю сохранить результат
'''

path_widget = widgets.Text(
    value='./result.xlsx',
    placeholder='Например: C:/temp/df_pair_list.xlsx или ./df_pair_list.xlsx',
    description='Путь:',
    layout=widgets.Layout(width='700px')
)

# Чекбокс: сохранять ли индекс
index_widget = widgets.Checkbox(
    value=False,
    description='Сохранять индекс'
)

# Кнопка экспорта
export_button = widgets.Button(
    description='Выгрузить в Excel',
    layout=widgets.Layout(width='20%', height='40px'),
    button_style='success',
    icon='download'
)

# Поле для сообщений
output = widgets.Output()

def export_to_excel(button):
    with output:
        output.clear_output()
        
        try:
            file_path = path_widget.value.strip()
            
            if not file_path:
                print("Ошибка: укажите путь к файлу.")
                return
            
            if not file_path.lower().endswith('.xlsx'):
                file_path += '.xlsx'
            
            # Создаем папку, если она указана и не существует
            dir_name = os.path.dirname(file_path)
            if dir_name:
                os.makedirs(dir_name, exist_ok=True)
            
            # Проверка, что df_pair_list существует
            if 'df_pair_list' not in globals():
                print("Ошибка: DataFrame df_pair_list не найден.")
                return
            
            # Экспорт в Excel
            df_pair_list.to_excel(file_path, index=index_widget.value)
            
            print(f"Файл успешно сохранён: {os.path.abspath(file_path)}")
        
        except Exception as e:
            print(f"Ошибка при сохранении файла: {e}")

export_button.on_click(export_to_excel)

ui = widgets.VBox([
    path_widget,
    index_widget,
    export_button,
    output
])

display(ui)

In [ ]:

# Для того чтобы запускать приложение в один клик, надо создать ярлык на рабочем столе и вписать туда такою строку: 

# C:\Windows\System32\WindowsPowerShell\v1.0\powershell.exe -NoExit -Command "C:\Users\rokotyev\AppData\Local\miniconda3\python.exe -m voila 'C:\Users\rokotyev\Yandex.Disk\_Main Data Rokotyan\1. IT\test.ipynb' --port=8866"  

# ВАЖНО! Есть ограничение по длинне символов. Проверь что путь до тетрадки в него.

------

5. ПРОСМОТР РЕЗУЛЬТАТОВ

In [ ]:
'''
Ячейка номер: 5.1. Назначение: Вспомогательные данные и функция для построения одного графика (для одной точки, одного KPI).

kpi_source_map - словарь, который для каждого KPI указывает откуда брать сырые недельные данные для графика: 
исходный датафрейм и название колонки со значением. Это те же самые исходные датафреймы, на основе которых 
считаются KPI для подбора точек (df_sales, df_osa, df_visits, df_facing_fact, df_picos, df_shelfshare, df_marketshare).
Для 'SALES TREND' используются те же данные, что и для 'kCAF' (продажи), для 'MS TREND' - те же, что и для 
'MARKET SHARE' (доля рынка), т.к. это трендовые версии этих же показателей.

kpi_column_in_pairlist - для "простых" KPI (не трендовых) указывает, в какой колонке df_pair_list лежит 
среднее значение показателя, рассчитанное для подбора точек (ячейка 3.1). Используется для отрисовки линии 
среднего на графике.

Функция plot_kpi_subplot рисует один Scatterplot для одной точки (SHIP_TO) и одного KPI на уже готовой оси (ax):
- по горизонтали - недели из выбранного пользователем периода (week_list из ячейки 2.3), расположенные по 
  порядку без "дыры" на границе годов (так же, как в слайдере периода)
- по вертикали - значение показателя за эту неделю
- если для недели есть несколько записей - берём среднее по неделе (функция get_weekly_series из ячейки 3.1)
- точки, которые попадают в выбросы по методу IQR (та же функция remove_outliers_iqr, что используется при 
  расчёте KPI в ячейке 3.1) закрашиваются красным, остальные - синим
- для 'kCAF', 'OSA', 'VISITS', 'PICOS', 'FACING', 'SHELF SHARE', 'MARKET SHARE' рисуется синяя штриховая 
  горизонтальная линия среднего значения (без учёта выбросов), взятая напрямую из df_pair_list (то самое 
  число, что реально использовалось при подборе контрольных точек). 
  ОБНОВЛЕНО: рядом с линией теперь подписано и само числовое значение среднего, округлённое до 2 знаков 
  после запятой.
- для 'SALES TREND' и 'MS TREND' поверх точек рисуется прямая линия тренда, посчитанная по очищенным от 
  выбросов данным тем же способом, что и SALES_SLOPE / MS_SLOPE в ячейке 3.1 (линия среднего для этих двух 
  KPI не рисуется - там уже есть линия тренда). 
  ОБНОВЛЕНО: рядом с линией тренда теперь подписан угол её наклона относительно горизонтальной оси, 
  округлённый до целых градусов.
'''

kpi_source_map = {
    'kCAF': (df_sales, 'kCAF'),
    'OSA': (df_osa, 'OSA'),
    'VISITS': (df_visits, 'PHOTO_AUDITS_PER_WEEK'),
    'PICOS': (df_picos, 'PICOS_SCORE_FACT'),
    'FACING': (df_facing_fact, 'GROUP_FACT'),
    'SHELF SHARE': (df_shelfshare, 'SHELF_SHARE'),
    'MARKET SHARE': (df_marketshare, 'MARKET_SHARE'),
    'SALES TREND': (df_sales, 'kCAF'),
    'MS TREND': (df_marketshare, 'MARKET_SHARE'),
}

kpi_column_in_pairlist = {
    'kCAF': 'kCAF',
    'OSA': 'OSA',
    'VISITS': 'PHOTO_AUDITS_PER_WEEK',
    'PICOS': 'PICOS_SCORE_FACT',
    'FACING': 'GROUP_FACT',
    'SHELF SHARE': 'SHELF_SHARE_AVG',
    'MARKET SHARE': 'MARKET_SHARE_AVG',
}

def plot_kpi_subplot(ax, ship_to, kpi_name, week_list):
    df_raw, value_col = kpi_source_map[kpi_name]
    se_week_list = pd.Series(map(int, week_list), name='YEAR_WEEK')

    df_point_raw = df_raw.loc[df_raw['SHIP_TO'] == ship_to]
    df_point_weekly = get_weekly_series(df_point_raw, se_week_list, value_col)
    weekly_series = df_point_weekly.set_index('YEAR_WEEK')[value_col].sort_index()

    ax.set_facecolor('white')

    if weekly_series.shape[0] == 0:
        ax.text(0.5, 0.5, 'Нет данных', ha='center', va='center', transform=ax.transAxes, fontsize=8)
        ax.set_xticks([])
        ax.set_yticks([])
        return

    # Определяем выбросы той же функцией, что использовалась при расчёте KPI (ячейка 3.1)
    clean_series = remove_outliers_iqr(weekly_series)
    is_outlier = ~weekly_series.index.isin(clean_series.index)

    x_pos = np.arange(len(weekly_series))  # позиции по оси X - порядковые номера недель (без "дыр")
    point_colors = np.where(is_outlier, 'red', 'blue')

    ax.scatter(x_pos, weekly_series.values, c=point_colors, s=14, zorder=3)

    # Линия среднего значения + подпись самого числа (округлено до 2 знаков после запятой)
    if kpi_name in kpi_column_in_pairlist:
        pair_col = kpi_column_in_pairlist[kpi_name]
        if pair_col in df_pair_list.columns:
            df_point_row = df_pair_list.loc[df_pair_list['SHIP_TO'] == ship_to]
            if df_point_row.shape[0] > 0 and pd.notna(df_point_row[pair_col].iloc[0]):
                mean_value = df_point_row[pair_col].iloc[0]
                mean_value = int(round(mean_value,0)) if mean_value//1 > 0 else round(mean_value,2)
                ax.axhline(y=mean_value, color='blue', linestyle='--', linewidth=1, zorder=2)
                ax.text(x_pos[-1], mean_value, f'{mean_value}', fontsize=10, color='black', fontweight ='bold', bbox ={'facecolor':'white', 'alpha':0.5},
                        ha='right', va='bottom', zorder=4)

    # Линия тренда + подпись угла наклона (округлено до целых градусов)
    if kpi_name in ('SALES TREND', 'MS TREND') and len(clean_series) >= 2:
        clean_x_pos = np.where(weekly_series.index.isin(clean_series.index))[0]
        fit_x = np.arange(len(clean_series))
        slope, intercept = np.polyfit(fit_x, clean_series.values, 1)
        predicted_y = intercept + slope * fit_x
        ax.plot(clean_x_pos, predicted_y, color='blue', linewidth=1.2, zorder=2)

        angle_deg = np.degrees(np.arctan(slope))
        ax.text(clean_x_pos[-1], predicted_y[-1], f'{angle_deg:.0f}°', fontsize=10, color='black', fontweight ='bold', bbox ={'facecolor':'white', 'alpha':0.5},
                ha='right', va='bottom', zorder=4)

    # Подписи недель по оси X - показываем не все подряд, чтобы не было каши на маленьком экране
    tick_step = max(1, len(weekly_series) // 4)
    ax.set_xticks(x_pos[::tick_step])
    ax.set_xticklabels(weekly_series.index[::tick_step], rotation=45, ha='right', fontsize=6)
    ax.tick_params(axis='y', labelsize=6)

In [ ]:
'''
Ячейка номер: 5.2. Назначение: Функция построения всех графиков для одной выбранной пары точек.

Строит сетку графиков: одна строка на каждый KPI, выбранный пользователем в ячейке 2.4, один столбец на 
каждую точку (первый столбец - тестовая точка, остальные - подобранные контрольные точки, в том порядке, 
в котором они были найдены). Название KPI подписано слева от первого графика в строке (как подпись оси Y).

Строки-заглушки с POS_TYPE = 'Control_POS_not_found' в построении графиков не участвуют - для них нет 
реальных данных (SHIP_TO = 0), поэтому показываются только фактически найденные контрольные точки.
'''

def draw_pair_charts(pair_number):
    output_charts.clear_output()
    with output_charts:
        if pair_number is None:
            return

        df_this_pair = df_pair_list.loc[df_pair_list['PAIR_NUMBER'] == pair_number]
        df_test = df_this_pair.loc[df_this_pair['POS_TYPE'].isin(['Test_POS', 'Test_POS_wo_pair'])]
        df_control = df_this_pair.loc[df_this_pair['POS_TYPE'] == 'Control_POS']

        if df_test.shape[0] == 0:
            print('Для этой пары не найдены данные тестовой точки.')
            return

        if len(kpi_list) == 0:
            print('В ячейке 2.4 не выбрано ни одного KPI - показывать нечего.')
            return

        points_ship_to = [df_test['SHIP_TO'].iloc[0]] + df_control['SHIP_TO'].tolist()
        points_labels = ['Тест: ' + str(points_ship_to[0])] + [f'Контроль {i+1}: {sid}' for i, sid in enumerate(df_control['SHIP_TO'].tolist())]

        n_points = len(points_ship_to)
        n_kpi = len(kpi_list)

        fig, axes = plt.subplots(nrows=n_kpi, ncols=n_points, figsize=(2.6 * n_points, 2.0 * n_kpi), squeeze=False)
        fig.patch.set_facecolor('white')

        for row, kpi_name in enumerate(kpi_list):
            axes[row, 0].set_ylabel(kpi_name, fontsize=8, fontweight='bold')
            for col, ship_to in enumerate(points_ship_to):
                ax = axes[row, col]
                plot_kpi_subplot(ax, ship_to, kpi_name, week_list)
                if row == 0:
                    ax.set_title(points_labels[col], fontsize=7)

        plt.tight_layout()
        plt.show()

In [ ]:
'''
Ячейка номер: 5.3. Назначение: Виджет выбора пары для просмотра результатов подбора.

Выпадающий список содержит все номера пар (PAIR_NUMBER) из df_pair_list, подпись каждого пункта списка 
дополнительно показывает SHIP_TO тестовой точки этой пары для удобства. При выборе пары ниже отрисовываются 
графики (функция draw_pair_charts из ячейки 5.2). Список пар и графики автоматически пересобираются 
(и прячутся) при повторном нажатии кнопки "Рассчитать" в ячейке 4.1.
'''

def get_pair_options():
    if df_pair_list.shape[0] == 0 or 'PAIR_NUMBER' not in df_pair_list.columns:
        return []
    options = []
    for pair_num in sorted(df_pair_list['PAIR_NUMBER'].unique()):
        df_test_row = df_pair_list.loc[(df_pair_list['PAIR_NUMBER'] == pair_num) & (df_pair_list['POS_TYPE'].isin(['Test_POS', 'Test_POS_wo_pair']))]
        if df_test_row.shape[0] > 0:
            ship_to_test = df_test_row['SHIP_TO'].iloc[0]
            label = f'Пара {pair_num} (тест: {ship_to_test})'
        else:
            label = f'Пара {pair_num}'
        options.append((label, pair_num))
    return options

dropdown_pair = widgets.Dropdown(
    options=get_pair_options(),
    description='Выберите пару:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px')
)

output_charts = widgets.Output()

def on_pair_change(change):
    if change['name'] == 'value':
        draw_pair_charts(change['new'])

dropdown_pair.observe(on_pair_change, names='value')

display(widgets.VBox([dropdown_pair, output_charts]))